## Dataset for testing Tim's permutation response criterion. 
### 22 oct 2025

Set up for importing to Matlab. Will eventually add the function to the ResponseCriterion class in this repo.

Varied parameters:
- number of trials
- baseline firing rate, varied using a thresholded exponential function
- gain in firing rate for response period, given as a multiple of the baseline firing rate that's randomly pulled from an interval
- duration of response, randomly pulled from an interval 

Spike times are scaled such that spikes == 0 occurred right at the "stimulus" period. 




In [130]:
import sys
sys.path.append("/home/al/Documents/code/generate_responses")

from pathlib import Path

import numpy as np
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.io import savemat

from tqdm import tqdm

from config_plot import *
from generators import PoissonSpikeGenerator
from stats import ResponseCriteria

path_save = Path("/media/al/sansidk/")

In [131]:
# simulation params
n_trial_range = [5, 10, 20, 30, 40, 50, 100, 150, 200, 300]
n_samples = 10_000

# response params
duration_range = [0.1, 0.2]
response_fr_ratio_range = [10, 20]

# baseline firing rate params
threshold = 2
scale = 3
rng = np.random.default_rng()

def determine_extra_trial_counts(n_trials, default=200, factor=5,):
    if n_trials < 50:
        n_supplement_trials = default
    else:
        n_supplement_trials = n_trials * factor
    return n_supplement_trials

In [119]:
# generate non-responsive spike trains

# preallocate params
n_NC = int(n_samples / 2)

latency      = 0.350
duration     = 0.1
baseline_T   = 2
stimulus_T   = 2
dt           = 0.001
induce_refractory_period = True

u = rng.uniform(size=n_NC)
fr_baseline = threshold - scale * np.log(u)

n_trial_baseline = np.random.choice(n_trial_range, n_NC)
n_supplement_trials = np.vectorize(determine_extra_trial_counts)(n_trial_baseline)

NC_rasters = [None] * n_NC
NC_supplement_trials = [None] * n_NC

df_nc = pd.DataFrame({
    "response": np.zeros(n_NC, dtype=np.int8),
    "fr_baseline": fr_baseline.astype(float),
    "fr_response": fr_baseline.astype(float),
    "latency": np.full(n_NC, latency, dtype=float),
    "duration": np.full(n_NC, duration, dtype=float),
    "time_baseline": np.full(n_NC, baseline_T, dtype=float),
    "time_stimulus": np.full(n_NC, stimulus_T, dtype=float),
    "refractory_period_induced": np.ones(n_NC, dtype=np.int8),
})

for i, fr in enumerate(tqdm(fr_baseline)):
    n_trials = n_trial_baseline[i]
    n_supplement_trials = determine_extra_trial_counts(n_trials)

    baseline_fr  = fr
    response_fr  = fr

    generator = PoissonSpikeGenerator(
    baseline_fr=baseline_fr,
    response_fr=response_fr,
    latency=latency,
    duration=duration,
    baseline_T=baseline_T,
    stimulus_T=stimulus_T,
    dt=dt,
    induce_refractory_period=induce_refractory_period
)

    # generate trials
    trial_activity = generator.generate(n_trials)
    supp_trial_activity = generator.generate(n_supplement_trials)

    # rescale
    trial_activity = [t - baseline_T for t in trial_activity]    
    supp_trial_activity = [t - baseline_T for t in supp_trial_activity]

    NC_rasters[i] = trial_activity
    NC_supplement_trials[i] = supp_trial_activity



100%|██████████| 5000/5000 [01:35<00:00, 52.62it/s]


In [120]:
df_nc["rasters"] = NC_rasters
df_nc["supp_rasters"] = NC_supplement_trials

In [121]:
# generate responsive spike trains

# preallocate params
n_C = int(n_samples / 2)

latency      = 0.350
baseline_T   = 2
stimulus_T   = 2
dt           = 0.001
induce_refractory_period = True

u = rng.uniform(size=n_C)
fr_baseline = threshold - scale * np.log(u)

ratios = np.random.randint(response_fr_ratio_range[0], response_fr_ratio_range[1], size=n_C)
fr_response = fr_baseline * ratios

durations = rng.uniform(duration_range[0], duration_range[1], size=n_C)

n_trial_responses = np.random.choice(n_trial_range, n_C)
n_supplement_trials = np.vectorize(determine_extra_trial_counts)(n_trial_responses)

C_rasters = [None] * n_C
C_supplement_trials = [None] * n_C

df_c = pd.DataFrame({
    "response": np.ones(n_C, dtype=np.int8),
    "fr_baseline": fr_baseline.astype(float),
    "fr_response": fr_response.astype(float),
    "latency": np.full(n_C, latency, dtype=float),
    "duration": durations.astype(float),
    "time_baseline": np.full(n_C, baseline_T, dtype=float),
    "time_stimulus": np.full(n_C, stimulus_T, dtype=float),
    "refractory_period_induced": np.ones(n_C, dtype=np.int8),
})

for i, fr in enumerate(tqdm(fr_baseline)):
    n_trials = n_trial_responses[i]
    n_supplement_trials = determine_extra_trial_counts(n_trials)

    baseline_fr  = fr
    response_fr  = fr_response[i]
    
    duration = durations[i]

    generator = PoissonSpikeGenerator(
    baseline_fr=baseline_fr,
    response_fr=response_fr,
    latency=latency,
    duration=duration,
    baseline_T=baseline_T,
    stimulus_T=stimulus_T,
    dt=dt,
    induce_refractory_period=induce_refractory_period
)

    # generate trials
    trial_activity = generator.generate(n_trials)
    supp_trial_activity = generator.generate(n_supplement_trials)

    # rescale
    trial_activity = [t - baseline_T for t in trial_activity]    
    supp_trial_activity = [t - baseline_T for t in supp_trial_activity]

    C_rasters[i] = trial_activity
    C_supplement_trials[i] = supp_trial_activity



100%|██████████| 5000/5000 [01:38<00:00, 50.52it/s]


In [122]:
df_c["rasters"] = C_rasters
df_c["supp_rasters"] = C_supplement_trials

In [ ]:
dataset = pd.concat((df_nc, df_c))
dataset_randomized = dataset.sample(frac=1, random_state=rng.integers(1e9)).reset_index(drop=True)


In [132]:
mat_dict = {col: dataset_randomized[col].to_numpy() for col in dataset_randomized.columns}

# Save
savemat(path_save / "2025-10-22_simulated_responses.mat", {"data": mat_dict})